Leyan Burait
1211439
ENCS5141_Assignment1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
%matplotlib inline
sns.set_style('whitegrid')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#lood datat
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('data.csv')
#print first 5 row
df.head()

In [ ]:
# cleane name of coulm
df.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num',
              'marital_status', 'occupation', 'relationship', 'race',
              'gender', 'capital_gain', 'capital_loss', 'hours_per_week',
              'native_country', 'income']

# display basic info
df.info()

In [ ]:
print(df.isnull().sum())

for col in df.columns:
    if df[col].dtype == 'object':
        print(f"{col}: {df[col].unique()}")
        if '?' in df[col].unique():
            print(f"?there are missing value {col}")

df.replace('?', np.nan, inplace=True)

# fully miss value
df['workclass'].fillna(df['workclass'].mode()[0], inplace=True)
df['occupation'].fillna(df['occupation'].mode()[0], inplace=True)
df['native_country'].fillna(df['native_country'].mode()[0], inplace=True)

In [ ]:
# ??????? ????????
print(df.describe())

# ???? ????? ?????
plt.figure(figsize=(10,6))
sns.histplot(df['age'], bins=30)
plt.title('age')
plt.show()

# ????? ????? ??????
plt.figure(figsize=(10,6))
sns.boxplot(x='income', y='age', data=df)
plt.title('age vs income')
plt.show()

# ????? ????? ????? ?????????
plt.figure(figsize=(10,6))
sns.histplot(df['hours_per_week'], bins=30)
plt.title('destrebute of week hours')
plt.show()

In [ ]:
# ????? ??????? ????? (income)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

# One-Hot Encoding ????????? ???????
categorical_cols = ['workclass', 'education', 'marital_status', 'occupation',
                   'relationship', 'race', 'gender', 'native_country']

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# ??? ???????? ??? ???????
df.head()

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# ??? ????????? ???????? ?????
X = df.drop('income', axis=1)
y = df['income']

# ?????? ???? 15 ????
selector = SelectKBest(score_func=f_classif, k=15)
X_new = selector.fit_transform(X, y)

# ?????? ??? ????? ??????? ????????
selected_features = X.columns[selector.get_support()]
print("??????? ????????:")
print(selected_features)

# ????? ???????? ???????? ????????
df = pd.concat([X[selected_features], y], axis=1)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ????? ????????
X = df.drop('income', axis=1)
y = df['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ????? ????? Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# ???????
y_pred = rf.predict(X_test)
print("??? ???????:", accuracy_score(y_test, y_pred))
print("\n????? ???????:")
print(classification_report(y_test, y_pred))

In [ ]:
# ??? ???????? ???????
df.to_csv('cleaneddata.csv', index=False)
files.download('cleaneddata.csv')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import time

# ????? ????????
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('cleaneddata.csv')

# ????? ???????? ??? ??????? ?????? ???
X = df.drop('income', axis=1)
y = df['income']

# ????? ???????? ??? ????? ??????? (70% ?????? 30% ??????)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# ????? Scaling (??? ???? ?? SVM ?MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ????? ????? Random Forest
rf = RandomForestClassifier(random_state=42)

# ?????? ???? ???????
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# ????? ?? ???? ?????? ???????? GridSearchCV
rf_grid = GridSearchCV(rf, rf_params, cv=5, scoring='accuracy')
start_time = time.time()
rf_grid.fit(X_train, y_train)
rf_time = time.time() - start_time

# ???? ??????
print("best  Random Forest:", rf_grid.best_params_)

# ??????? ??? ?????? ????????
rf_best = rf_grid.best_estimator_
y_pred_rf = rf_best.predict(X_test)

# ???? ????????
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("\nRandom Forest:")
print("accuracy:", rf_accuracy)
print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("F1-score:", rf_f1)
print("training time:", rf_time, "?????")

In [ ]:
# ????? ????? SVM
svm = SVC(random_state=42)

# ?????? ???? ???????
svm_params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# ????? ?? ???? ??????
svm_grid = GridSearchCV(svm, svm_params, cv=5, scoring='accuracy')
start_time = time.time()
svm_grid.fit(X_train_scaled, y_train)
svm_time = time.time() - start_time

# ???? ??????
print("best SVM:", svm_grid.best_params_)

# ??????? ??? ?????? ????????
svm_best = svm_grid.best_estimator_
y_pred_svm = svm_best.predict(X_test_scaled)

# ???? ????????
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)

print("\nSVM:")
print("accuracy:", svm_accuracy)
print("Precision:", svm_precision)
print("Recall:", svm_recall)
print("F1-score:", svm_f1)
print("time traning : ", svm_time, "?????")

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

# Define MLP with better convergence parameters
mlp = MLPClassifier(random_state=42,
                   max_iter=1000,
                   early_stopping=True,
                   learning_rate='adaptive',
                   n_iter_no_change=10)

# Expanded parameter grid
mlp_params = {
    'hidden_layer_sizes': [(50,), (100,), (50,50), (100,50)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# Grid search
mlp_grid = GridSearchCV(mlp, mlp_params, cv=5, scoring='accuracy', n_jobs=-1)
start_time = time.time()
mlp_grid.fit(X_train_scaled, y_train)
mlp_time = time.time() - start_time

# Best parameters
print("Best MLP parameters:", mlp_grid.best_params_)

# Evaluation
mlp_best = mlp_grid.best_estimator_
y_pred_mlp = mlp_best.predict(X_test_scaled)

mlp_accuracy = accuracy_score(y_test, y_pred_mlp)
mlp_precision = precision_score(y_test, y_pred_mlp)
mlp_recall = recall_score(y_test, y_pred_mlp)
mlp_f1 = f1_score(y_test, y_pred_mlp)

print("\nMLP Performance:")
print("Accuracy:", mlp_accuracy)
print("Precision:", mlp_precision)
print("Recall:", mlp_recall)
print("F1-score:", mlp_f1)
print("Training time:", mlp_time, "seconds")

In [ ]:
# ????? ???? ????????
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'SVM', 'MLP'],
    'Accuracy': [rf_accuracy, svm_accuracy, mlp_accuracy],
    'Precision': [rf_precision, svm_precision, mlp_precision],
    'Recall': [rf_recall, svm_recall, mlp_recall],
    'F1-score': [rf_f1, svm_f1, mlp_f1],
    'Training Time (s)': [rf_time, svm_time, mlp_time]
})

# ??? ???? ????????
print( " compare model's :")
print(comparison)

# ???? ????????
plt.figure(figsize=(12, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
x = np.arange(len(metrics))
width = 0.25

plt.bar(x - width, comparison.iloc[0, 1:5], width, label='Random Forest')
plt.bar(x, comparison.iloc[1, 1:5], width, label='SVM')
plt.bar(x + width, comparison.iloc[2, 1:5], width, label='MLP')

plt.xlabel('Scale')
plt.ylabel('value')
plt.title('comper model')
plt.xticks(x, metrics)
plt.legend()
plt.show()

# ?????? ??? ???????
plt.figure(figsize=(8, 5))
plt.bar(comparison['Model'], comparison['Training Time (s)'])
plt.xlabel('model')
plt.ylabel('  traning time (sec)')
plt.title(' comper all model of traning time   ')
plt.show()

In [ ]:
# Results Analysis
print("\nAnalysis and Conclusions:")

# Find the best model by accuracy
best_model = comparison.loc[comparison['Accuracy'].idxmax()]
print(f"\nThe best model in terms of accuracy is {best_model['Model']} with accuracy {best_model['Accuracy']:.4f}")

# Find the fastest model
fastest_model = comparison.loc[comparison['Training Time (s)'].idxmin()]
print(f"The fastest model is {fastest_model['Model']} with training time {fastest_model['Training Time (s)']:.2f} seconds")

# Comprehensive analysis
print("\nGeneral Observations:")
print("- Random Forest achieved the best overall performance but with longer training time")
print("- SVM was the fastest but had lower accuracy")
print("- MLP achieved a good balance between performance and training time")

# Recommendations
print("\nRecommendations:")
print("1. If accuracy is the priority, Random Forest is preferred")
print("2. If speed is important, SVM can be used with some compromise on accuracy")
print("3. MLP is a good choice for achieving a balance between performance and speed")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import time

# ????? ???????? ??????? ?? ????? ?????
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('/content/cleaned_income_data.csv')

In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
%matplotlib inline
sns.set_style('whitegrid')

#  Load and Inspect Dataset
# Load dataset
df = pd.read_csv('/content/data.csv')

# Print column names to debug
print("Dataset Columns:", df.columns.tolist())

# Check for any leading/trailing whitespace in column names
df.columns = df.columns.str.strip()

# Display basic information
print("\nDataset Info:")
print(df.info())
print("\nFirst 5 Rows:")
print(df.head())

# Verify expected categorical columns
expected_categorical_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']
missing_cols = [col for col in expected_categorical_cols if col not in df.columns]
if missing_cols:
    print(f"\nWarning: The following expected columns are missing: {missing_cols}")
else:
    print("\nAll expected categorical columns are present.")

# Data Cleaning
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)
print("\nMissing Values (NaN):")
print(df.isnull().sum())

# Impute missing values for categorical columns with mode
for col in expected_categorical_cols:
    if col in df.columns and df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

# Verify no missing values remain
print("\nMissing Values After Imputation:")
print(df.isnull().sum())

#  Feature Engineering
# Encode target variable
le = LabelEncoder()
if 'income' in df.columns:
    df['income'] = le.fit_transform(df['income'])
else:
    raise ValueError("Target column 'income' not found in the dataset.")

# Define categorical and numerical columns
categorical_cols = [col for col in expected_categorical_cols if col in df.columns]
numerical_cols = ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
numerical_cols = [col for col in numerical_cols if col in df.columns]  # Ensure numerical columns exist

print("\nCategorical Columns Used:", categorical_cols)
print("Numerical Columns Used:", numerical_cols)

# One-hot encode categorical features
if categorical_cols:
    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
else:
    print("No categorical columns available for one-hot encoding.")
    df_encoded = df.copy()

# Scale numerical features
scaler = StandardScaler()
if numerical_cols:
    df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])
else:
    print("No numerical columns available for scaling.")

# Verify encoded DataFrame
print("\nEncoded DataFrame Info:")
print(df_encoded.info())
print("\nFirst 5 Rows of Encoded DataFrame:")
print(df_encoded.head())

#  Feature Selection and Dimensionality Reduction
# Prepare features and target
X = df_encoded.drop('income', axis=1) if 'income' in df_encoded.columns else df_encoded
y = df_encoded['income'] if 'income' in df_encoded.columns else None

if y is None:
    raise ValueError("Target variable 'income' not found in encoded DataFrame.")

# Feature selection using SelectKBest
selector = SelectKBest(score_func=f_classif, k=min(20, X.shape[1]))  # Select top 20 features or fewer
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()
print("\nSelected Features:", selected_features)

# Dimensionality reduction using PCA
pca = PCA(n_components=0.95)  # Retain 95% variance
X_pca = pca.fit_transform(X_selected)
print("\nPCA Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Number of PCA Components:", X_pca.shape[1])

#  Save Preprocessed Data (Optional)
# Save preprocessed data for further use
df_encoded.to_csv('preprocessed_data.csv', index=False)
print("\nPreprocessed data saved to 'preprocessed_data.csv'.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import time
%matplotlib inline
sns.set_style('whitegrid')
le = LabelEncoder()
df['income'] = le.fit_transform(df['income'])  # <=50K -> 0, >50K -> 1

# One-hot encode categorical features
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Scale numerical features
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

# Feature selection using SelectKBest
X = df_encoded.drop('income', axis=1)
y = df_encoded['income']
selector = SelectKBest(score_func=f_classif, k=20)  # Select top 20 features
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()
print("\nSelected Features:", selected_features)

# Dimensionality reduction using PCA
pca = PCA(n_components=0.95)  # Retain 95% variance
X_pca = pca.fit_transform(X_selected)
print("\nPCA Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Number of PCA Components:", X_pca.shape[1])

In [ ]:
df_raw = pd.read_csv('/content/data.csv')
df_raw.replace('?', np.nan, inplace=True)
df_raw.dropna(inplace=True)  # Drop missing values for simplicity in raw data
df_raw['income'] = le.fit_transform(df_raw['income'])
df_raw_encoded = pd.get_dummies(df_raw, columns=categorical_cols, drop_first=True)

X_raw = df_raw_encoded.drop('income', axis=1)
y_raw = df_raw_encoded['income']
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Split preprocessed data
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

# Train Random Forest on raw data
rf_raw = RandomForestClassifier(random_state=42)
start_time = time.time()
rf_raw.fit(X_train_raw, y_train_raw)
rf_raw_time = time.time() - start_time
y_pred_raw = rf_raw.predict(X_test_raw)
rf_raw_accuracy = accuracy_score(y_test_raw, y_pred_raw)
rf_raw_precision = precision_score(y_test_raw, y_pred_raw)
rf_raw_recall = recall_score(y_test_raw, y_pred_raw)
rf_raw_f1 = f1_score(y_test_raw, y_pred_raw)

# Train Random Forest on preprocessed data
rf_preprocessed = RandomForestClassifier(random_state=42)
start_time = time.time()
rf_preprocessed.fit(X_train, y_train)
rf_preprocessed_time = time.time() - start_time
y_pred_preprocessed = rf_preprocessed.predict(X_test)
rf_preprocessed_accuracy = accuracy_score(y_test, y_pred_preprocessed)
rf_preprocessed_precision = precision_score(y_test, y_pred_preprocessed)
rf_preprocessed_recall = recall_score(y_test, y_pred_preprocessed)
rf_preprocessed_f1 = f1_score(y_test, y_pred_preprocessed)

# Compare raw vs. preprocessed
comparison_rf = pd.DataFrame({
    'Data': ['Raw', 'Preprocessed'],
    'Accuracy': [rf_raw_accuracy, rf_preprocessed_accuracy],
    'Precision': [rf_raw_precision, rf_preprocessed_precision],
    'Recall': [rf_raw_recall, rf_preprocessed_recall],
    'F1-score': [rf_raw_f1, rf_preprocessed_f1],
    'Training Time (s)': [rf_raw_time, rf_preprocessed_time]
})
print("\nRandom Forest: Raw vs. Preprocessed Data:")
print(comparison_rf)

# Visualize comparison
plt.figure(figsize=(10, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
x = np.arange(len(metrics))
width = 0.35
plt.bar(x - width/2, comparison_rf.iloc[0, 1:5], width, label='Raw Data')
plt.bar(x + width/2, comparison_rf.iloc[1, 1:5], width, label='Preprocessed Data')
plt.xlabel('Metrics')
plt.ylabel('Values')
plt.title('Random Forest: Raw vs. Preprocessed Data')
plt.xticks(x, metrics)
plt.legend()
plt.show()

In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=5, scoring='accuracy')
start_time = time.time()
rf_grid.fit(X_train, y_train)
rf_grid_time = time.time() - start_time
print("\nRandom Forest Best Parameters:", rf_grid.best_params_)
print("Random Forest Best Accuracy:", rf_grid.best_score_)

# Parameter tuning for SVM
svm_param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear']
}
svm_grid = GridSearchCV(SVC(random_state=42), svm_param_grid, cv=5, scoring='accuracy')
start_time = time.time()
svm_grid.fit(X_train, y_train)
svm_grid_time = time.time() - start_time
print("\nSVM Best Parameters:", svm_grid.best_params_)
print("SVM Best Accuracy:", svm_grid.best_score_)

# Parameter tuning for MLP
mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'learning_rate_init': [0.001, 0.01]
}
mlp_grid = GridSearchCV(MLPClassifier(random_state=42, max_iter=500), mlp_param_grid, cv=5, scoring='accuracy')
start_time = time.time()
mlp_grid.fit(X_train, y_train)
mlp_grid_time = time.time() - start_time
print("\nMLP Best Parameters:", mlp_grid.best_params_)
print("MLP Best Accuracy:", mlp_grid.best_score_)